In [1]:
from hydrafloods_langgraph_agent import GRAPH

d:\Conda\envs\floodagent\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
query = "Help me extract the flooded areas in the flood event in 2023 in the city of Venice, Italy. I want to know the flooded areas and the severity of the flood in those areas."

In [3]:
initial_state = {
    "query": query,
    "environment":  {"token_tracking_enabled": True,
                      "debug_trace_enabled": True}
}

In [4]:
result = GRAPH.invoke(initial_state)

[HYDRAFLOODS_TRACE] {"marker": "SATGPT_TRACE_DELETE_ME", "step": 1, "node": "detect_environment", "phase": "enter", "payload": {"query": "Help me extract the flooded areas in the flood event in 2023 in the city of Venice, Italy. I want to know the flooded areas and the severity of the flood in those areas."}}
[HYDRAFLOODS_TRACE] {"marker": "SATGPT_TRACE_DELETE_ME", "step": 2, "node": "detect_environment", "phase": "exit", "payload": {"gee_project_id": "configured", "llm_model": "gpt-4o-mini", "token_tracking_enabled": true, "debug_trace_enabled": true}}
[HYDRAFLOODS_TRACE] {"marker": "SATGPT_TRACE_DELETE_ME", "step": 3, "node": "parse_request", "phase": "enter", "payload": {"query": "Help me extract the flooded areas in the flood event in 2023 in the city of Venice, Italy. I want to know the flooded areas and the severity of the flood in those areas."}}
[HYDRAFLOODS_TRACE] {"marker": "SATGPT_TRACE_DELETE_ME", "step": 4, "node": "parse_request", "phase": "exit", "payload": {"parsed_requ

EEException: Image.unmask: If one image has no bands, the other must also have no bands. Got 0 and 1.

In [ ]:
result

{'query': 'Help me extract the flooded areas in the flood event in 2023 in the city of Venice, Italy. I want to know the flooded areas and the severity of the flood in those areas.',
 'environment': {'gee_project_id': 'configured',
  'llm_model': 'gpt-4o-mini',
  'token_tracking_enabled': True},
 'parsed_request': {'selected_tool': 'get_flood_extent_tile',
  'dataset': 'Sentinel1',
  'dates': ['2023-01-01', '2023-12-31'],
  'bbox': [12.3155, 45.4372, 12.4534, 45.4408],
  'algorithm': 'edge_otsu',
  'reference': 'occurrence',
  'action': 'flood_extent',
  'missing_fields': [],
  'heuristic_hints': {'action': 'describe_tools',
   'dataset': None,
   'dates': [],
   'bbox': None,
   'algorithm': 'edge_otsu',
   'reference': 'seasonal'}},
 'selected_tool': 'get_flood_extent_tile',
 'tool_result': {'status': 'ok',
  'tool_name': 'get_flood_extent_tile',
  'summary': '完成洪水范围提取并生成在线地图图层。',
  'inputs': {'dataset': 'Sentinel1',
   'start_date': '2023-01-01',
   'end_date': '2023-12-31',
   'bbo

: 

In [7]:
tiles = "https://earthengine.googleapis.com/v1/projects/flood-agent/maps/d77a2da6123ecb80ec5ae44616be4710-3fc4d3e1c2fe7949d244b7b896ce8363/tiles/{z}/{x}/{y}"

In [10]:
import folium
map = folium.Map(zoom_start=6)
folium.TileLayer(
    tiles=tiles,
    attr="Google Earth Engine",
    name="Water Mask",
    overlay=True,
    control=True
).add_to(map)

In [11]:
display(map)

In [12]:
water_img.visualize({
    "min":0,"max":1,
    "palette":"silver,navy",
    "dimensions":1500,
    "region":region
})

In [1]:
# T 

In [4]:
import ee
import hydrafloods as hf
from ee_utils import init_gee

init_gee()
region = ee.Geometry.Rectangle([90.3, 23.6, 90.5, 23.8])
dataset = hf.Sentinel1(region, '2020-07-01', '2020-07-30')


In [9]:
dataset._collection.first().getInfo()

{'type': 'Image',
 'bands': [{'id': 'VV',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'dimensions': [28959, 21550],
   'crs': 'EPSG:32646',
   'crs_transform': [10, 0, 99375.37575662392, 0, -10, 2678141.201458545]},
  {'id': 'VH',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'dimensions': [28959, 21550],
   'crs': 'EPSG:32646',
   'crs_transform': [10, 0, 99375.37575662392, 0, -10, 2678141.201458545]},
  {'id': 'angle',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [21, 10],
   'crs': 'EPSG:32646',
   'crs_transform': [12771.171769593493,
    -3478.3016659275163,
    132471.66781883157,
    2187.0629587895237,
    20184.762455353513,
    2462649.858977335]}],
 'version': 1776068450127364,
 'id': 'COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20200701T120425_20200701T120450_033261_03DA82_47FD',
 'properties': {'SNAP_Graph_Processing_Framework_GPF_vers': '7.0.3',
  'SLC_Processing_facility_org': 'ESA',
  'SLC_Processing_facili

In [ ]:
working = dataset.apply_func(hf.vv_vh_ratio)
water = working.apply_func(hf.edge_otsu, band='ratio', initial_threshold=-16, scale=150, invert=False, thresh_no_data=-16)
water_img = water.collection.mode().set('system:time_start', ee.Date('2020-07-15').millis())
water_img = water_img.clip(region)
print(water_img.getMapId({'min': 0, 'max': 1, 'palette': 'white,blue'})['tile_fetcher'].url_format)